01. Extracción de ZIP y filtrado estricto por Dengue

In [ ]:
import os
import zipfile
import glob
import pandas as pd
from pathlib import Path
import shutil

NOTEBOOK_DIR = Path.cwd()
ROOT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = ROOT_DIR / 'data'

archivo_zip = DATA_DIR / 'datos_al_ecosistema.zip'
carpeta_destino = ROOT_DIR / 'datos_extraidos'
archivo_salida = ROOT_DIR / 'consolidado_enfermedades_bucaramanga.xlsx'

if archivo_zip.exists():
    if carpeta_destino.exists():
        shutil.rmtree(carpeta_destino)

    print(f"1. Descomprimiendo archivos desde: {archivo_zip}")
    with zipfile.ZipFile(archivo_zip, 'r') as zip_ref:
        zip_ref.extractall(carpeta_destino)
    print("   ¡Descompresión completada con éxito!")
else:
    print(f"1. No se encontró el ZIP raw en: {archivo_zip}")
    print("   Continuaremos con los datos procesados disponibles en data/ si existen.")

# ==========================================
# 2. BUSCAR ARCHIVOS EXCEL EXCLUSIVAMENTE DE DENGUE
# ==========================================
patrones_dengue = ['*dengue*', '*Dengue*', '*DENGUE*']
archivos_xlsx = []

if carpeta_destino.exists():
    for patron in patrones_dengue:
        archivos_xlsx += glob.glob(str(carpeta_destino / '**' / patron / '*.xlsx'), recursive=True)
        archivos_xlsx += glob.glob(str(carpeta_destino / '**' / patron / '*.xls'), recursive=True)
        archivos_xlsx += glob.glob(str(carpeta_destino / patron / '*.xlsx'), recursive=True)
        archivos_xlsx += glob.glob(str(carpeta_destino / patron / '*.xls'), recursive=True)
        archivos_xlsx += glob.glob(str(carpeta_destino / '**' / f'{patron}.xlsx'), recursive=True)
        archivos_xlsx += glob.glob(str(carpeta_destino / '**' / f'{patron}.xls'), recursive=True)

    archivos_xlsx = list(set(archivos_xlsx))

print(f"\n2. ¡Se detectaron un total de {len(archivos_xlsx)} archivos Excel correspondientes a Dengue / Dengue Grave!")

if len(archivos_xlsx) == 0:
    print("   [!] Advertencia: No se encontraron carpetas o archivos que coincidan con la palabra 'Dengue'. Verifica los nombres del ZIP o usa dataset.xlsx ya generado.")

# ==========================================
# 3. PROCESAR, FILTRAR Y COMPILAR ENFERMEDADES
# ==========================================
lista_df_filtrados = []

palabras_clave_enfermedad = ['nombre_eve', 'nom_eve', 'evento', 'enfermedad', 'diag', 'cie10', 'causa', 'patol', 'nom_evento']

for i, archivo in enumerate(archivos_xlsx, start=1):
    nombre_breve = os.path.relpath(archivo, carpeta_destino)
    print(f"   [{i}/{len(archivos_xlsx)}] Procesando: {nombre_breve}...")

    df = None
    for motor in ['openpyxl', 'xlrd', None]:
        try:
            df = pd.read_excel(archivo, engine=motor)
            break
        except Exception:
            continue

    if df is None:
        try:
            lista_tablas = pd.read_html(archivo)
            if lista_tablas:
                df = lista_tablas[0]
        except Exception:
            print(f"      -> ERROR: No se pudo abrir este archivo.")
            continue

    try:
        df_trabajo = df.copy()
        columnas_minusculas = [str(c).strip().lower() for c in df_trabajo.columns]

        cols_depto = [df_trabajo.columns[idx] for idx, c in enumerate(columnas_minusculas) if 'departamen' in c or 'dep' in c]
        cols_muni = [df_trabajo.columns[idx] for idx, c in enumerate(columnas_minusculas) if 'municipio' in c or 'muni' in c]
        cols_enf = [df_trabajo.columns[idx] for idx, c in enumerate(columnas_minusculas) if any(p in c for p in palabras_clave_enfermedad)]

        if cols_depto and cols_muni:
            filtro_depto = df_trabajo[cols_depto].astype(str).apply(lambda x: x.str.upper().str.contains('SANTANDER')).any(axis=1)
            filtro_muni = df_trabajo[cols_muni].astype(str).apply(lambda x: x.str.upper().str.contains('BUCARAMANGA')).any(axis=1)

            df_filtrado = df_trabajo[filtro_depto & filtro_muni].copy()

            if not df_filtrado.empty:
                if cols_enf:
                    col_enfermedad_real = cols_enf[0]
                    df_filtrado['ENFERMEDAD_CONSOLIDADA'] = df_filtrado[col_enfermedad_real].astype(str).str.upper().str.strip()
                else:
                    df_filtrado['ENFERMEDAD_CONSOLIDADA'] = 'DENGUE (COLUMNA NO DETECTADA)'

                df_filtrado['ARCHIVO_ORIGEN'] = nombre_breve
                lista_df_filtrados.append(df_filtrado)
                print(f"      -> ¡Éxito! Se extrajeron {len(df_filtrado)} casos.")
            else:
                print('      -> No se hallaron casos para Bucaramanga en este archivo.')
        else:
            print('      -> Advertencia: No se hallaron columnas de departamento o municipio.')

    except Exception as e:
        print(f"      -> Error al procesar datos: {e}")

# ==========================================
# 4. EXPORTACIÓN Y GENERACIÓN DEL CONSOLIDADO
# ==========================================
print('\n3. Consolidando resultados finales...')
if lista_df_filtrados:
    df_final = pd.concat(lista_df_filtrados, ignore_index=True)
    df_final.columns = [str(c).upper() for c in df_final.columns]

    df_consolidado = df_final.groupby(['ENFERMEDAD_CONSOLIDADA']).size().reset_index(name='TOTAL_CASOS')
    df_consolidado = df_consolidado.sort_values(by='TOTAL_CASOS', ascending=False)

    with pd.ExcelWriter(archivo_salida, engine='openpyxl') as writer:
        df_consolidado.to_excel(writer, sheet_name='Consolidado_Enfermedades', index=False)
        df_final.to_excel(writer, sheet_name='Casos_Detallados', index=False)

    print(f"\n¡Proceso Terminado con éxito! Archivo guardado como: '{archivo_salida}'")
    print(f"Total de registros consolidados en Bucaramanga: {len(df_final)}")
    print(f"Diferentes tipos de dengue/variables detectadas: {len(df_consolidado)}")
else:
    print('\nNo se logró consolidar ningún registro de Dengue con los criterios especificados.')
    print('   Si no hay archivos brutos, usa dataset.xlsx o dataset_limpio.xlsx desde data/.')


1. Descomprimiendo archivos desde: c:\Users\USUARIO\OneDrive\Documentos\trabajos uni\IA_PREDICTIVA\data\datos_al_ecosistema.zip
   ¡Descompresión completada con éxito!

2. ¡Se detectaron un total de 30 archivos Excel correspondientes a Dengue / Dengue Grave!
   [1/30] Procesando: datos_al_ecosistema\Dengue_2010_2024\Datos_2013_210.xlsx...
      -> ¡Éxito! Se extrajeron 12035 casos.
   [2/30] Procesando: datos_al_ecosistema\Dengue_grave_2010_2024\Datos_2023_220.xlsx...
      -> ¡Éxito! Se extrajeron 59 casos.
   [3/30] Procesando: datos_al_ecosistema\Dengue_grave_2010_2024\Datos_2018_220.xls...
      -> ERROR: No se pudo abrir este archivo.
   [4/30] Procesando: datos_al_ecosistema\Dengue_grave_2010_2024\Datos_2012_220.xls...
      -> ERROR: No se pudo abrir este archivo.
   [5/30] Procesando: datos_al_ecosistema\Dengue_2010_2024\Datos_2015_210.xlsx...
      -> ¡Éxito! Se extrajeron 4111 casos.
   [6/30] Procesando: datos_al_ecosistema\Dengue_grave_2010_2024\Datos_2015_220.xls...
      

02.Preprocesamiento de tipos de datos, fechas y duplicados

In [5]:
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT_DIR / 'data'
INTERIM_DIR = DATA_DIR / '02_intermediate'

# Garantiza que estas variables SIEMPRE existan, incluso si алgo falla más abajo.
epidemiological_df = globals().get('epidemiological_df', pd.DataFrame())
precipitation_df = globals().get('precipitation_df', pd.DataFrame())
temperature_df = globals().get('temperature_df', pd.DataFrame())
if not isinstance(epidemiological_df, pd.DataFrame):
    epidemiological_df = pd.DataFrame()
if not isinstance(precipitation_df, pd.DataFrame):
    precipitation_df = pd.DataFrame()
if not isinstance(temperature_df, pd.DataFrame):
    temperature_df = pd.DataFrame()

# Fallback: si las variables no están en memoria (kernel distinto al de 01_EDA),
# se intenta cargar primero desde los archivos guardados por 01_EDA_exploracion_datos.ipynb
# en data/02_intermediate/ (epidemiological_df.xlsx, precipitation_df.csv, temperature_df.csv),
# y si no existen, desde dataset_limpio.xlsx en data/03_primary/.
if epidemiological_df.empty:
    try:
        if (INTERIM_DIR / 'epidemiological_df.xlsx').exists():
            epidemiological_df = pd.read_excel(INTERIM_DIR / 'epidemiological_df.xlsx')
            print('✅ epidemiological_df cargado desde data/02_intermediate/epidemiological_df.xlsx (generado por 01_EDA).')
        elif (DATA_DIR / '03_primary' / 'dataset_limpio.xlsx').exists():
            epidemiological_df = pd.read_excel(DATA_DIR / '03_primary' / 'dataset_limpio.xlsx')
            print('⚠️ epidemiological_df no estaba definido. Se cargó dataset_limpio.xlsx como fallback.')
    except Exception as e:
        print(f'❌ ERROR cargando epidemiological_df: {e}')
        epidemiological_df = pd.DataFrame()

for nombre in ['precipitation_df', 'temperature_df']:
    if globals().get(nombre) is None or not isinstance(globals().get(nombre), pd.DataFrame) or globals()[nombre].empty:
        csv_path = INTERIM_DIR / f'{nombre}.csv'
        try:
            if csv_path.exists():
                globals()[nombre] = pd.read_csv(csv_path)
                print(f'✅ {nombre} cargado desde data/02_intermediate/{nombre}.csv (generado por 01_EDA).')
        except Exception as e:
            print(f'❌ ERROR cargando {nombre}: {e}')

# ==========================================
# 0. DEFENSIVE CHECK & COLUMN CLEANING
# ==========================================
if epidemiological_df.empty:
    print('❌ ERROR: epidemiological_df is empty! Check your ZIP file paths or the fallback dataset in data/.')
else:
    epidemiological_df.columns = epidemiological_df.columns.str.strip().str.upper()

# ==========================================
# 1. LIMPIEZA DE EPIDEMIOLOGICAL_DF
# ==========================================
if not epidemiological_df.empty:
    if 'INI_SIN' in epidemiological_df.columns:
        epidemiological_df['INI_SIN'] = pd.to_datetime(epidemiological_df['INI_SIN'], errors='coerce')
    else:
        print("❌ WARNING: 'INI_SIN' column still not found. Available columns are:", epidemiological_df.columns.tolist())
else:
    print('❌ WARNING: epidemiological_df está vacío. Se omite la limpieza de epidemiological_df.')

# ==========================================
# 2. LIMPIEZA DE PRECIPITATION_DF
# ==========================================
if not precipitation_df.empty and 'FechaObservacion' in precipitation_df.columns:
    precipitation_df['FechaObservacion'] = pd.to_datetime(precipitation_df['FechaObservacion'], errors='coerce')
    precipitation_df['ValorObservado'] = (
        precipitation_df['ValorObservado']
        .astype(str)
        .str.replace(',', '.', regex=False)
    )
    precipitation_df['ValorObservado'] = pd.to_numeric(precipitation_df['ValorObservado'], errors='coerce')
    for col in ['Latitud', 'Longitud']:
        if col in precipitation_df.columns:
            precipitation_df[col] = (
                precipitation_df[col]
                .astype(str)
                .str.replace(',', '.', regex=False)
                .replace({'nan': None})
            )
            precipitation_df[col] = pd.to_numeric(precipitation_df[col], errors='coerce')

    if {'FechaObservacion', 'ValorObservado', 'Latitud', 'Longitud'}.issubset(precipitation_df.columns):
        precipitation_df.dropna(subset=['FechaObservacion', 'ValorObservado', 'Latitud', 'Longitud'], inplace=True)
    print('Precipitación - Registros antes de duplicados:', len(precipitation_df))
    precipitation_df = precipitation_df.drop_duplicates(subset=['CodigoEstacion', 'FechaObservacion', 'ValorObservado']) if {'CodigoEstacion', 'FechaObservacion', 'ValorObservado'}.issubset(precipitation_df.columns) else precipitation_df
    print('Precipitación - Registros después de duplicados:', len(precipitation_df))
    if 'FechaObservacion' in precipitation_df.columns:
        precipitation_df['AÑO'] = precipitation_df['FechaObservacion'].dt.isocalendar().year.astype(int)
        precipitation_df['SEMANA_EPI'] = precipitation_df['FechaObservacion'].dt.isocalendar().week.astype(int)
else:
    print('⚠️ precipitation_df no está disponible o faltan columnas requeridas para limpieza.')

# ==========================================
# 3. LIMPIEZA DE TEMPERATURE_DF
# ==========================================
if not temperature_df.empty and 'FechaObservacion' in temperature_df.columns:
    temperature_df['FechaObservacion'] = pd.to_datetime(temperature_df['FechaObservacion'], errors='coerce')
    temperature_df['ValorObservado'] = (
        temperature_df['ValorObservado']
        .astype(str)
        .str.replace(',', '.', regex=False)
    )
    temperature_df['ValorObservado'] = pd.to_numeric(temperature_df['ValorObservado'], errors='coerce')
    for col in ['Latitud', 'Longitud']:
        if col in temperature_df.columns:
            temperature_df[col] = (
                temperature_df[col]
                .astype(str)
                .str.replace(',', '.', regex=False)
                .replace({'nan': None})
            )
            temperature_df[col] = pd.to_numeric(temperature_df[col], errors='coerce')

    if {'FechaObservacion', 'ValorObservado', 'Latitud', 'Longitud'}.issubset(temperature_df.columns):
        temperature_df.dropna(subset=['FechaObservacion', 'ValorObservado', 'Latitud', 'Longitud'], inplace=True)
    print('Temperatura - Registros antes de duplicados:', len(temperature_df))
    temperature_df = temperature_df.drop_duplicates(subset=['CodigoEstacion', 'FechaObservacion', 'ValorObservado']) if {'CodigoEstacion', 'FechaObservacion', 'ValorObservado'}.issubset(temperature_df.columns) else temperature_df
    print('Temperatura - Registros después de duplicados:', len(temperature_df))
    if 'FechaObservacion' in temperature_df.columns:
        temperature_df['AÑO'] = temperature_df['FechaObservacion'].dt.isocalendar().year.astype(int)
        temperature_df['SEMANA_EPI'] = temperature_df['FechaObservacion'].dt.isocalendar().week.astype(int)
else:
    print('⚠️ temperature_df no está disponible o faltan columnas requeridas para limpieza.')

# ==========================================
# 4. VERIFICACIÓN DEL ESTADO FINAL
# ==========================================
print('\n=== Epidemiological DataFrame Info ===')
if not epidemiological_df.empty:
    epidemiological_df.info()
else:
    print('epidemiological_df está vacío.')

print('\n=== Precipitation DataFrame Info ===')
if not precipitation_df.empty:
    precipitation_df.info()
else:
    print('precipitation_df está vacío o no definido.')

print('\n=== Temperature DataFrame Info ===')
if not temperature_df.empty:
    temperature_df.info()
else:
    print('temperature_df está vacío o no definido.')

Precipitación - Registros antes de duplicados: 608152
Precipitación - Registros después de duplicados: 590151
Temperatura - Registros antes de duplicados: 73669
Temperatura - Registros después de duplicados: 71393

=== Epidemiological DataFrame Info ===
<class 'pandas.DataFrame'>
Index: 15997 entries, 0 to 43728
Columns: 193 entries, CONSECUTIVE to TEMPERATURA_RANGO_ROLLMEAN8
dtypes: datetime64[us](1), float64(123), int64(37), object(6), str(26)
memory usage: 23.7+ MB

=== Precipitation DataFrame Info ===
<class 'pandas.DataFrame'>
Index: 590151 entries, 0 to 608007
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   CodigoEstacion     590151 non-null  int64         
 1   CodigoSensor       590151 non-null  int64         
 2   FechaObservacion   590151 non-null  datetime64[us]
 3   ValorObservado     590151 non-null  float64       
 4   NombreEstacion     590151 non-null  str           
 5

03.Ingeniería de variables (Lags, Master Grid, Rolling windows) y cruce final

In [6]:
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT_DIR / 'data'
INTERIM_DIR = DATA_DIR / '02_intermediate'

# Primero se reutiliza lo que haya en memoria (si esta celda corre justo
# después del Paso 2 en el mismo kernel). Si no está disponible, se cae a
# los archivos guardados en data/02_intermediate/ en vez de depender de la
# "memoria" entre notebooks que existía en Colab.
precipitation_df = globals().get('precipitation_df', pd.DataFrame())
temperature_df = globals().get('temperature_df', pd.DataFrame())
epidemiological_df = globals().get('epidemiological_df', pd.DataFrame())

if not isinstance(precipitation_df, pd.DataFrame) or precipitation_df.empty:
    csv_path = INTERIM_DIR / 'precipitation_df.csv'
    if csv_path.exists():
        precipitation_df = pd.read_csv(csv_path)
        print('✅ precipitation_df cargado desde data/02_intermediate/precipitation_df.csv.')
    else:
        precipitation_df = pd.DataFrame()

if not isinstance(temperature_df, pd.DataFrame) or temperature_df.empty:
    csv_path = INTERIM_DIR / 'temperature_df.csv'
    if csv_path.exists():
        temperature_df = pd.read_csv(csv_path)
        print('✅ temperature_df cargado desde data/02_intermediate/temperature_df.csv.')
    else:
        temperature_df = pd.DataFrame()

if not isinstance(epidemiological_df, pd.DataFrame) or epidemiological_df.empty:
    xlsx_path = INTERIM_DIR / 'epidemiological_df.xlsx'
    if xlsx_path.exists():
        epidemiological_df = pd.read_excel(xlsx_path)
        print('✅ epidemiological_df cargado desde data/02_intermediate/epidemiological_df.xlsx.')
    else:
        epidemiological_df = pd.DataFrame()

# Asegura que ValorObservado/Latitud/Longitud sean numéricos sin importar si
# el DataFrame vino de memoria (ya limpio en el Paso 2) o del CSV crudo en
# disco (donde el decimal puede venir con coma, ej. "23,5").
for _clima_df in (precipitation_df, temperature_df):
    if _clima_df.empty:
        continue
    for _col in ['ValorObservado', 'Latitud', 'Longitud']:
        if _col in _clima_df.columns and not pd.api.types.is_numeric_dtype(_clima_df[_col]):
            _clima_df[_col] = pd.to_numeric(
                _clima_df[_col].astype(str).str.replace(',', '.', regex=False),
                errors='coerce'
            )

raw_climate_available = (
    isinstance(precipitation_df, pd.DataFrame) and not precipitation_df.empty and
    isinstance(temperature_df, pd.DataFrame) and not temperature_df.empty and
    'FechaObservacion' in precipitation_df.columns and
    'FechaObservacion' in temperature_df.columns
)

if not raw_climate_available:
    print('⚠️ No hay datos raw de precipitación/temperatura disponibles. Se omite la generación de variables climáticas desde raw data.')
    if epidemiological_df.empty:
        print('❌ epidemiological_df también está vacío. No se puede continuar con este bloque.')
else:
    if 'INI_SIN' not in epidemiological_df.columns or epidemiological_df.empty:
        print("Detected epidemiological_df is not in detailed format or is empty. Attempting to reload from 'Casos_Detallados' sheet.")
        try:
            epidemiological_df = pd.read_excel(
                DATA_DIR / 'consolidado_enfermedades_bucaramanga.xlsx',
                sheet_name='Casos_Detallados'
            )
            epidemiological_df.columns = epidemiological_df.columns.str.strip().str.upper()
            print(f"Epidemiological_df reloaded con {len(epidemiological_df)} rows and {len(epidemiological_df.columns)} columns.")
        except Exception as e:
            print(f"Error reloading epidemiological_df from 'Casos_Detallados': {e}")
            print("Please ensure 'consolidado_enfermedades_bucaramanga.xlsx' exists in data/ and contains a 'Casos_Detallados' sheet.")

    def crear_variables_climaticas_avanzadas(clima_df, master_grid, variable, es_precipitacion=True, n_lags=8):
        if es_precipitacion:
            agregaciones = {
                f"{variable}_Total": ("ValorObservado", "sum"),
                f"{variable}_Max10min": ("ValorObservado", "max"),
                f"{variable}_Std": ("ValorObservado", "std"),
                "_Registros": ("ValorObservado", "count"),
                "_Lluvias": ("ValorObservado", lambda x: (x > 0).sum())
            }
        else:
            agregaciones = {
                f"{variable}_Media": ("ValorObservado", "mean"),
                f"{variable}_Min": ("ValorObservado", "min"),
                f"{variable}_Max": ("ValorObservado", "max"),
                f"{variable}_Std": ("ValorObservado", "std")
            }

        weekly_station = (
            clima_df.groupby(["CodigoEstacion", "Departamento", "Municipio", "AÑO", "SEMANA_EPI"], as_index=False)
            .agg(**agregaciones)
        )

        if es_precipitacion:
            weekly_station[f"{variable}_Frecuencia"] = (
                weekly_station["_Lluvias"] / weekly_station["_Registros"].replace(0, np.nan)
            )
            weekly_station = weekly_station.drop(columns=["_Registros", "_Lluvias"])

        metricas = [c for c in weekly_station.columns if c not in ["CodigoEstacion", "Departamento", "Municipio", "AÑO", "SEMANA_EPI"]]

        weekly_local = (
            weekly_station.groupby(["Departamento", "Municipio", "AÑO", "SEMANA_EPI"], as_index=False)
            .agg({m: "mean" for m in metricas})
        )

        weekly_dept = (
            weekly_station.groupby(["Departamento", "AÑO", "SEMANA_EPI"], as_index=False)
            .agg({m: "mean" for m in metricas})
        )

        weekly = master_grid.merge(weekly_local, on=["Departamento", "Municipio", "AÑO", "SEMANA_EPI"], how="left")
        weekly[f"{variable}_Sensor_Disponible"] = weekly[metricas[0]].notna().astype(int)
        weekly = weekly.merge(weekly_dept, on=["Departamento", "AÑO", "SEMANA_EPI"], how="left", suffixes=("", "_dep"))

        for m in metricas:
            weekly[m] = weekly[m].fillna(weekly.get(f"{m}_dep", np.nan))
            if es_precipitacion:
                weekly[m] = weekly[m].fillna(0)

        if not es_precipitacion:
            weekly[f"{variable}_Rango"] = weekly[f"{variable}_Max"] - weekly[f"{variable}_Min"]
            metricas.append(f"{variable}_Rango")

        weekly = weekly.sort_values(["Departamento", "Municipio", "AÑO", "SEMANA_EPI"]).reset_index(drop=True)
        metricas_originales = metricas.copy()

        for m in metricas_originales:
            for window in [2, 4, 8]:
                col_roll_mean = f"{m}_RollMean{window}"
                weekly[col_roll_mean] = (
                    weekly.groupby(["Departamento", "Municipio"])[m]
                    .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
                )
                metricas.append(col_roll_mean)

        if es_precipitacion:
            m_total = f"{variable}_Total"
            if m_total in metricas_originales:
                for window in [2, 4, 8]:
                    col_roll_sum = f"{m_total}_RollSum{window}"
                    weekly[col_roll_sum] = (
                        weekly.groupby(["Departamento", "Municipio"])[m_total]
                        .transform(lambda s: s.shift(1).rolling(window, min_periods=1).sum())
                    )
                    weekly[col_roll_sum] = weekly[col_roll_sum].fillna(0)
                    metricas.append(col_roll_sum)

            weekly['is_wet_week'] = (weekly[f"{variable}_Total"] > 0).astype(int)
            weekly['Precipitacion_WetWeeks_RollSum8'] = (
                weekly.groupby(["Departamento", "Municipio"])["is_wet_week"]
                .transform(lambda s: s.shift(1).rolling(8, min_periods=1).sum())
            )
            weekly['Precipitacion_WetWeeks_RollSum8'] = weekly['Precipitacion_WetWeeks_RollSum8'].fillna(0)
            metricas.append('Precipitacion_WetWeeks_RollSum8')
            weekly = weekly.drop(columns=['is_wet_week'])

        columnas_finales = ["Departamento", "Municipio", "AÑO", "SEMANA_EPI", f"{variable}_Sensor_Disponible"]
        columnas_finales.extend(metricas_originales)

        for m in metricas_originales:
            for lag in range(1, n_lags + 1):
                col_lag = f"{m}_L{lag}"
                weekly[col_lag] = weekly.groupby(["Departamento", "Municipio"])[m].shift(lag)
                if es_precipitacion:
                    weekly[col_lag] = weekly[col_lag].fillna(0)
                else:
                    weekly[col_lag] = weekly.groupby(["Departamento", "Municipio"])[col_lag].transform(lambda x: x.bfill().ffill())
                columnas_finales.append(col_lag)

        columnas_finales.extend([m for m in metricas if m not in metricas_originales])
        return weekly[list(dict.fromkeys(columnas_finales))]

    cols_borrar = [
        c for c in epidemiological_df.columns
        if (c.startswith("Precipitacion") or c.startswith("Temperatura") or
            c.endswith("_x") or c.endswith("_y") or c == "FechaSemana")
    ]
    epidemiological_df = epidemiological_df.drop(columns=cols_borrar, errors="ignore")

    def normalizar(texto):
        if pd.isna(texto):
            return "DESCONOCIDO"
        return "".join(c for c in unicodedata.normalize("NFD", str(texto).upper().strip()) if unicodedata.category(c) != "Mn")

    precipitation_df.columns = [c.replace("\x11", "N") for c in precipitation_df.columns]
    temperature_df.columns = [c.replace("\x11", "N") for c in temperature_df.columns]
    epidemiological_df.columns = [c.replace("\x11", "N") for c in epidemiological_df.columns]

    epidemiological_df["INI_SIN"] = pd.to_datetime(epidemiological_df["INI_SIN"], errors="coerce")
    precipitation_df["FechaObservacion"] = pd.to_datetime(precipitation_df["FechaObservacion"], errors="coerce")
    temperature_df["FechaObservacion"] = pd.to_datetime(temperature_df["FechaObservacion"], errors="coerce")

    iso_e = epidemiological_df["INI_SIN"].dt.isocalendar()
    epidemiological_df["AÑO_INI_SIN"] = iso_e.year.fillna(0).astype(int)
    epidemiological_df["SEMANA_EPI_INI_SIN"] = iso_e.week.fillna(0).astype(int)

    iso_p = precipitation_df["FechaObservacion"].dt.isocalendar()
    precipitation_df["AÑO"], precipitation_df["SEMANA_EPI"] = iso_p.year.astype(int), iso_p.week.astype(int)

    iso_t = temperature_df["FechaObservacion"].dt.isocalendar()
    temperature_df["AÑO"], temperature_df["SEMANA_EPI"] = iso_t.year.astype(int), iso_t.week.astype(int)

    precipitation_df["Municipio"] = precipitation_df["Municipio"].apply(normalizar)
    precipitation_df["Departamento"] = precipitation_df["Departamento"].apply(normalizar)
    temperature_df["Municipio"] = temperature_df["Municipio"].apply(normalizar)
    temperature_df["Departamento"] = temperature_df["Departamento"].apply(normalizar)
    epidemiological_df["MUNICIPIO_OCURRENCIA"] = epidemiological_df["MUNICIPIO_OCURRENCIA"].apply(normalizar)
    epidemiological_df["DEPARTAMENTO_OCURRENCIA"] = epidemiological_df["DEPARTAMENTO_OCURRENCIA"].apply(normalizar)
    epidemiological_df["NOMBRE_EVENTO"] = epidemiological_df["NOMBRE_EVENTO"].apply(normalizar)

    epidemiological_df = epidemiological_df[
        (epidemiological_df["MUNICIPIO_OCURRENCIA"] == "BUCARAMANGA") &
        (epidemiological_df["NOMBRE_EVENTO"] == "DENGUE")
    ].copy()

    precipitation_df = precipitation_df[
        (precipitation_df["Departamento"] == "SANTANDER") & (precipitation_df["Municipio"] == "BUCARAMANGA")
    ].copy()

    temperature_df = temperature_df[
        (temperature_df["Departamento"] == "SANTANDER") & (temperature_df["Municipio"] == "BUCARAMANGA")
    ].copy()

    overall_min_date = min(epidemiological_df['INI_SIN'].min(), precipitation_df['FechaObservacion'].min(), temperature_df['FechaObservacion'].min())
    overall_max_date = max(epidemiological_df['INI_SIN'].max(), precipitation_df['FechaObservacion'].max(), temperature_df['FechaObservacion'].max())

    calendar_dates = pd.date_range(start=overall_min_date, end=overall_max_date, freq='W-MON')
    calendar_iso = calendar_dates.to_series().dt.isocalendar()

    master_grid = pd.DataFrame({
        "AÑO": calendar_iso.year.astype(int),
        "SEMANA_EPI": calendar_iso.week.astype(int)
    }).drop_duplicates().sort_values(["AÑO", "SEMANA_EPI"]).reset_index(drop=True)

    master_grid["Departamento"] = "SANTANDER"
    master_grid["Municipio"] = "BUCARAMANGA"
    master_grid = master_grid[["Departamento", "Municipio", "AÑO", "SEMANA_EPI"]]

    print(f"📙 Universo optimizado para Bucaramanga:")
    print(f"  - Casos de estudio INS (Recalculados): {len(epidemiological_df)}")
    print(f"  - Semanas cronológicas reales en Master Grid: {len(master_grid)}\n")

    weekly_precipitation = crear_variables_climaticas_avanzadas(
        precipitation_df, master_grid, variable="Precipitacion", es_precipitacion=True, n_lags=8
    )

    weekly_temperature = crear_variables_climaticas_avanzadas(
        temperature_df, master_grid, variable="Temperatura", es_precipitacion=False, n_lags=8
    )

    llaves_izq = ["DEPARTAMENTO_OCURRENCIA", "MUNICIPIO_OCURRENCIA", "AÑO_INI_SIN", "SEMANA_EPI_INI_SIN"]
    llaves_der = ["Departamento", "Municipio", "AÑO", "SEMANA_EPI"]

    epidemiological_df = epidemiological_df.merge(
        weekly_precipitation, left_on=llaves_izq, right_on=llaves_der, how="left"
    ).drop(columns=llaves_der, errors="ignore")

    epidemiological_df = epidemiological_df.merge(
        weekly_temperature, left_on=llaves_izq, right_on=llaves_der, how="left"
    ).drop(columns=llaves_der, errors="ignore")

    epidemiological_df = epidemiological_df.dropna(subset=["Precipitacion_Total", "Temperatura_Media"])

    print("✅ Pipeline robusto, unificado y autocontenido completado con éxito.")
    print(f"Estructura final: {epidemiological_df.shape[0]} filas y {epidemiological_df.shape[1]} columnas optimizadas para LightGBM.")
    display(epidemiological_df.head())

📙 Universo optimizado para Bucaramanga:
  - Casos de estudio INS (Recalculados): 15997
  - Semanas cronológicas reales en Master Grid: 745

✅ Pipeline robusto, unificado y autocontenido completado con éxito.
Estructura final: 15997 filas y 307 columnas optimizadas para LightGBM.


,CONSECUTIVE,COD_EVE,FEC_NOT,SEMANA,ANO,COD_PRE,COD_SUB,EDAD,UNI_MED,NACIONALIDAD,...,Temperatura_Min_RollMean8,Temperatura_Max_RollMean2,Temperatura_Max_RollMean4,Temperatura_Max_RollMean8,Temperatura_Std_RollMean2,Temperatura_Std_RollMean4,Temperatura_Std_RollMean8,Temperatura_Rango_RollMean2,Temperatura_Rango_RollMean4,Temperatura_Rango_RollMean8
0,7233578.0,210,2020-03-08,11,2020,6800100701,2,11,1,170,...,19.0875,31.40,31.275,31.1625,3.187714,3.195656,3.155348,12.05,11.975,12.0750
1,7233589.0,210,2020-03-04,10,2020,6827601366,13,39,1,170,...,18.9250,32.30,31.550,31.2000,3.421829,3.223863,3.192037,13.05,12.550,12.2750
2,7233675.0,210,2020-02-14,7,2020,6800100701,3,50,1,170,...,19.0750,30.80,30.925,30.7625,3.025897,3.009725,3.077109,12.05,11.900,11.6875
3,7233824.0,210,2020-05-12,19,2020,6830700720,1,12,1,170,...,19.3875,30.50,30.725,30.3625,2.989465,2.931671,2.852754,10.95,11.125,10.9750
4,7234680.0,210,2020-07-24,30,2020,6800103418,1,24,1,170,...,18.8625,30.25,30.350,30.4750,3.010670,2.997781,3.020153,11.30,11.300,11.6125


04.Procesamiento y limpieza de las proyecciones del DANE (Histórico y Futuro)

In [2]:
import pandas as pd
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT_DIR / 'data'
INTERIM_DIR = DATA_DIR / '02_intermediate'
RAW_DIR = DATA_DIR / '01_raw'

def localizar_archivo_dane(nombre_archivo):
    # Busca primero en 02_intermediate (donde los moviste ahora) y si no
    # está ahí, cae a 01_raw (ubicación original) o a data/ directamente.
    for carpeta in (INTERIM_DIR, RAW_DIR, DATA_DIR):
        ruta = carpeta / nombre_archivo
        if ruta.exists():
            return ruta
    return DATA_DIR / nombre_archivo

ruta_2005_2017 = localizar_archivo_dane('DCD-area-proypoblacion-Mun-2005-2017_VP.xlsx')
ruta_2018_2042 = localizar_archivo_dane('PPED-AreaMun-2018-2042_VP.xlsx')

bga_2005_2017 = pd.DataFrame()
bga_2018_2042 = pd.DataFrame()

# Función maestra para procesar y validar proyecciones del DANE
def procesar_proyeccion_dane(df, nombre_etapa, rango_esperado, col_poblacion_original):
    print(f"\n{'='*50}")
    print(f"--- Procesando Dataset: {nombre_etapa} ---")
    print(f"{'='*50}")

    # 1. Homogeneizar la columna de población a 'Población'
    if col_poblacion_original in df.columns and col_poblacion_original != 'Población':
        df = df.rename(columns={col_poblacion_original: 'Población'})

    # 2. Limpieza y estandarización estricta de columnas clave
    if 'MPIO' in df.columns:
        df['MPIO_CLEAN'] = (
            df['MPIO']
            .astype(str)
            .str.replace(r'\\.0$', '', regex=True)
            .str.strip()
            .str.zfill(5)
        )
    else:
        raise ValueError("Error crítico: La columna 'MPIO' no se encuentra en el dataset.")

    if 'ÁREA GEOGRÁFICA' in df.columns:
        df['ÁREA GEOGRÁFICA_CLEAN'] = (
            df['ÁREA GEOGRÁFICA']
            .astype(str)
            .str.upper()
            .str.strip()
        )
    else:
        raise ValueError("Error crítico: La columna 'ÁREA GEOGRÁFICA' no se encuentra en el dataset.")

    # 3. Filtrado específico para Bucaramanga (Código 68001) y Área "TOTAL"
    df_bga = df[
        (df['MPIO_CLEAN'] == '68001') &
        (df['ÁREA GEOGRÁFICA_CLEAN'] == 'TOTAL')
    ].copy()

    # Seleccionar solo AÑO y Población, luego ordenar
    df_bga = df_bga[['AÑO', 'Población']].sort_values('AÑO').reset_index(drop=True)

    # 4. Asegurar tipo numérico y cálculos demográficos básicos
    df_bga["Población"] = pd.to_numeric(df_bga["Población"], errors="coerce")

    AREA = 165.11  # km²
    df_bga["densidad_poblacional"] = df_bga["Población"] / AREA

    print(f"Rango de años detectado: {df_bga['AÑO'].min()} hasta {df_bga['AÑO'].max()}")
    print(f"Filas procesadas para Bucaramanga: {df_bga.shape[0]}")

    # 5. Validaciones de Calidad
    # Validación A: Unicidad
    try:
        assert df_bga["AÑO"].is_unique, "¡Error! Existen años duplicados."
        print("✅ Unicidad: Exitosa (No hay años duplicados).")
    except AssertionError as e:
        print(f"❌ {e}")
        display(df_bga[df_bga.duplicated("AÑO", keep=False)])

    # Validación B: Cronología esperada
    años_reales = set(df_bga["AÑO"].dropna())
    años_faltantes = set(rango_esperado) - años_reales

    if len(años_faltantes) == 0:
        print("✅ Cronología: Exitosa (Serie continua en el rango esperado).")
    else:
        print(f"⚠️ Hay años faltantes en el rango esperado: {años_faltantes}")

    # Validación C: Población válida (> 0)
    try:
        assert (df_bga["Población"] > 0).all(), "¡Error! Población <= 0 o nula."
        print("✅ Numérica: Exitosa (Todas las poblaciones > 0).")
    except AssertionError as e:
        print(f"❌ {e}")
        display(df_bga[df_bga["Población"].isna() | (df_bga["Población"] <= 0)])

    print("\nVista previa del resultado:")
    print(df_bga.head(3))

    return df_bga

if ruta_2005_2017.exists() and ruta_2018_2042.exists():
    # Archivo 1: 2005 - 2017
    df_antiguo_crudo = pd.read_excel(ruta_2005_2017, header=0)
    bga_2005_2017 = procesar_proyeccion_dane(
        df=df_antiguo_crudo,
        nombre_etapa="Histórico 2005 - 2017",
        rango_esperado=range(2005, 2018),
        col_poblacion_original="Población"
    )

    # Archivo 2: 2018 - 2042
    df_nuevo_crudo = pd.read_excel(ruta_2018_2042, header=0, skiprows=[1, 2])
    bga_2018_2042 = procesar_proyeccion_dane(
        df=df_nuevo_crudo,
        nombre_etapa="Proyecciones 2018 - 2042",
        rango_esperado=range(2018, 2043),
        col_poblacion_original="TOTAL"
    )
else:
    bga_2005_2017 = pd.DataFrame()
    bga_2018_2042 = pd.DataFrame()
    print(f'⚠️ No se encontraron archivos DANE. Buscados en {INTERIM_DIR}, {RAW_DIR} y {DATA_DIR}. Se omite el procesamiento de las proyecciones históricas y futuras.')


--- Procesando Dataset: Histórico 2005 - 2017 ---
Rango de años detectado: 2005 hasta 2017
Filas procesadas para Bucaramanga: 13
✅ Unicidad: Exitosa (No hay años duplicados).
✅ Cronología: Exitosa (Serie continua en el rango esperado).
✅ Numérica: Exitosa (Todas las poblaciones > 0).

Vista previa del resultado:
    AÑO  Población  densidad_poblacional
0  2005     484962           2937.205499
1  2006     492693           2984.028829
2  2007     500260           3029.858882

--- Procesando Dataset: Proyecciones 2018 - 2042 ---
Rango de años detectado: 2018 hasta 2042
Filas procesadas para Bucaramanga: 25
✅ Unicidad: Exitosa (No hay años duplicados).
✅ Cronología: Exitosa (Serie continua en el rango esperado).
✅ Numérica: Exitosa (Todas las poblaciones > 0).

Vista previa del resultado:
    AÑO  Población  densidad_poblacional
0  2018     581138           3519.702017
1  2019     590543           3576.664042
2  2020     601668           3644.043365


05.Unificación de la serie demográfica

In [3]:
# 4. Unificar las proyecciones de ambos períodos
dane_bga_unificado = pd.concat([bga_2005_2017, bga_2018_2042], ignore_index=True)

if dane_bga_unificado.empty:
    print('⚠️ dane_bga_unificado está vacío. No se pudo unificar la serie demográfica.')
else:
    print("\n--- Serie Unificada de Población para Bucaramanga (2005-2042) ---")
    print(f"Rango de años: {dane_bga_unificado['AÑO'].min()} - {dane_bga_unificado['AÑO'].max()}")
    print(f"Total de filas: {dane_bga_unificado.shape[0]}")

    print("\nPrimeras 5 filas del DataFrame unificado:")
    display(dane_bga_unificado.head())

    display(dane_bga_unificado.tail())
    print("\nÚltimas 5 filas del DataFrame unificado:")


--- Serie Unificada de Población para Bucaramanga (2005-2042) ---
Rango de años: 2005 - 2042
Total de filas: 38

Primeras 5 filas del DataFrame unificado:


,AÑO,Población,densidad_poblacional
0,2005,484962,2937.205499
1,2006,492693,2984.028829
2,2007,500260,3029.858882
3,2008,507605,3074.344376
4,2009,514867,3118.327176


,AÑO,Población,densidad_poblacional
33,2038,615966,3730.640179
34,2039,614908,3724.232330
35,2040,613712,3716.988674
36,2041,612361,3708.806250
37,2042,610880,3699.836473



Últimas 5 filas del DataFrame unificado:


06.Cálculo de deltas poblacionales para el año 2018

In [6]:
from pathlib import Path

if dane_bga_unificado.empty:
    print('⚠️ dane_bga_unificado está vacío. No se pueden calcular deltas poblacionales.')
else:
    # Deltas año contra año en TODA la serie 2005-2042 (no solo 2018).
    # El caso de 2018 originalmente se calculaba aparte porque es la fila
    # donde empalman los dos archivos DANE (histórico 2005-2017 + proyección
    # 2018-2042); al ordenar por AÑO y unificar ambos en un solo DataFrame,
    # un shift(1) año a año ya resuelve ese empalme igual que el resto de
    # la serie, así que no hace falta tratarlo aparte.
    dane_bga_unificado = dane_bga_unificado.sort_values('AÑO').reset_index(drop=True)

    poblacion_prev = dane_bga_unificado['Población'].shift(1)
    densidad_prev = dane_bga_unificado['densidad_poblacional'].shift(1)

    dane_bga_unificado['crecimiento_poblacional'] = (
        (dane_bga_unificado['Población'] - poblacion_prev) / poblacion_prev
    ) * 100
    dane_bga_unificado['incremento_habitantes'] = dane_bga_unificado['Población'] - poblacion_prev
    dane_bga_unificado['densidad_delta'] = dane_bga_unificado['densidad_poblacional'] - densidad_prev
    # El primer año (2005) no tiene año anterior, así que sus deltas quedan
    # en NaN correctamente (no hay con qué compararlo).

    display(dane_bga_unificado[(dane_bga_unificado['AÑO'] >= 2017) & (dane_bga_unificado['AÑO'] <= 2019)])
    print(f"Deltas calculados para {dane_bga_unificado['crecimiento_poblacional'].notna().sum()} de {len(dane_bga_unificado)} años.")

    # Persistir dane_bga_unificado para que la celda "07. Limpieza principal"
    # pueda cargarlo desde disco (cada notebook/kernel corre por separado,
    # no comparten memoria como en Colab).
    ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    INTERIM_DIR = ROOT_DIR / 'data' / '02_intermediate'
    INTERIM_DIR.mkdir(parents=True, exist_ok=True)
    dane_bga_unificado.to_excel(INTERIM_DIR / 'dane_bga_unificado.xlsx', index=False)
    print(f'\n✅ dane_bga_unificado guardado en {INTERIM_DIR / "dane_bga_unificado.xlsx"}')

,AÑO,Población,densidad_poblacional,crecimiento_poblacional,incremento_habitantes,densidad_delta
12,2017,570253,3453.776270,1.446303,8130.0,49.239901
13,2018,581138,3519.702017,1.908802,10885.0,65.925746
14,2019,590543,3576.664042,1.618376,9405.0,56.962025


Deltas calculados para 37 de 38 años.

✅ dane_bga_unificado guardado en /home/agustine/Downloads/IA_PREDICTIVA/data/02_intermediate/dane_bga_unificado.xlsx


07.Limpieza principal del DataFrame de Dengue Grave, Filtrado de Variables y Merge

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT_DIR / 'data'
INTERIM_DIR = DATA_DIR / '02_intermediate'
RAW_DIR = DATA_DIR / '01_raw'
PRIMARY_DIR = DATA_DIR / '03_primary'

# ================================
# Cargar archivo
# ================================
candidatos = [
    RAW_DIR / 'epidemiological_data_bucaramanga_processed_dengue_grave1.xlsx',
    DATA_DIR / 'epidemiological_data_bucaramanga_processed_dengue_grave1.xlsx',
    PRIMARY_DIR / 'dataset_limpio.xlsx',
    DATA_DIR / 'dataset_limpio.xlsx',
]
ruta = next((c for c in candidatos if c.exists()), None)
if ruta is None:
    raise FileNotFoundError(f'No se encontró ningún archivo procesado en {RAW_DIR}, {PRIMARY_DIR} ni {DATA_DIR}.')
if ruta.name == 'dataset_limpio.xlsx':
    print(f'⚠️ No se encontró el archivo procesado original. Se usará {ruta} como fallback.')
else:
    print(f'✅ Cargando {ruta}')

df = pd.read_excel(ruta)
palabras_eliminar = [

    # Identificadores
    "consecutive", "cod_eve", "cod_pre", "cod_sub", "cod_ase", "archivo_origen", "particion",

    # Fechas
    "fec_", "fecha_nto",

    # Países y ubicación
    "pais", "nacionalidad", "nombre_nacionalidad", "departamento", "municipio", "cod_pais", "cod_dpto", "cod_mun",

    # Datos personales
    "sexo", "edad", "ocupacion", "estrato", "area", "tip_ss", "uni_med",

    # Poblaciones especiales
    "gp_", "per_etn", "gru_pob", "nom_grupo",

    # Fuerzas militares
    "fm_",

    # Variables que generan fuga de información
    "tip_cas", "pac_hos", "con_fin", "fec_hos", "fec_def", "estado_final_de_caso", "nom_est_f_caso",
    "confirmados", "ajuste", "cer_def", "cbmte",

    # Otras administrativas
    "fuente", "va_sispro"
]

# ================================
# Variables del modelo
# ================================
variables_modelo = [
    col for col in df.columns
    if (
        col.lower() == "nom_upgd"

        # Variables temporales
        or col.lower() in [
            "nombre_evento", "semana", "ano", "año_ini_sin", "semana_epi_ini_sin", "ini_sin", "semana_sin", "semana_cos"
        ]

        # Variables epidemiológicas
        or col.lower().startswith("persistencia_alerta")
        or col.lower().startswith("alerta_lag")
        or col.lower().startswith("casos_")
        or col.lower().startswith("media_casos")
        or col.lower().startswith("std_casos")

        # Variables climáticas
        or col.lower().startswith("precipitacion")
        or col.lower().startswith("temperatura")
    )
]

# Conservar únicamente las variables del modelo
df = df[variables_modelo]

# ================================
# Columnas climáticas
# ================================
columnas_clima = [
    c for c in df.columns
    if c.lower().startswith("precipitacion")
    or c.lower().startswith("temperatura")
]

filas_antes = len(df)
df = df.dropna(subset=columnas_clima)
filas_despues = len(df)

print(f"Filas antes: {filas_antes:,}")
print(f"Filas después: {filas_despues:,}")
print(f"Filas eliminadas: {filas_antes - filas_despues:,}")

# ======================================
# Cargar variables demográficas DANE
# ======================================
dane_path = INTERIM_DIR / 'dane_bga_unificado.xlsx'
if not dane_path.exists():
    dane_path = DATA_DIR / 'dane_bga_unificado.xlsx'

if dane_path.exists():
    dane = pd.read_excel(dane_path)
    print('DANE cargado desde', dane_path)
else:
    dane = pd.DataFrame()
    print(f'⚠️ No se encontró dane_bga_unificado.xlsx en {INTERIM_DIR} ni en {DATA_DIR}. Se omitirá el merge DANE.')

if not dane.empty:
    print(dane.columns)
    df = df.merge(
        dane,
        left_on="ANO",
        right_on="AÑO",
        how="left",
        validate="many_to_one"
    )
    if 'AÑO' in df.columns:
        df.drop(columns="AÑO", inplace=True)
else:
    print('⚠️ El merge DANE se omitió porque el DataFrame DANE está vacío.')

# ============================================================
# Reemplazar 0 por NaN SOLO en variables donde 0 es imposible
# ============================================================
columnas_invalidar = []
for col in df.columns:
    nombre = col.lower()
    if nombre.startswith("temperatura_media") or nombre.startswith("temperatura_max") or \
       nombre.startswith("temperatura_min") or nombre.startswith("temperatura_std"):
        columnas_invalidar.append(col)

if columnas_invalidar:
    df[columnas_invalidar] = df[columnas_invalidar].replace(0, np.nan)

print(f"Columnas revisadas: {len(columnas_invalidar)}")

# ================================
# Guardar resultado
# ================================
PRIMARY_DIR.mkdir(parents=True, exist_ok=True)
salida = PRIMARY_DIR / 'dataset_modelo_sin_nan_clima.xlsx'
df.to_excel(salida, index=False)
print(f"\nArchivo guardado como: {salida}")

✅ Cargando /home/agustine/Downloads/IA_PREDICTIVA/data/01_raw/epidemiological_data_bucaramanga_processed_dengue_grave1.xlsx
Filas antes: 15,342
Filas después: 15,321
Filas eliminadas: 21
DANE cargado desde /home/agustine/Downloads/IA_PREDICTIVA/data/02_intermediate/dane_bga_unificado.xlsx
Index(['AÑO', 'Población', 'densidad_poblacional', 'crecimiento_poblacional',
       'incremento_habitantes', 'densidad_delta'],
      dtype='str')
Columnas revisadas: 48

Archivo guardado como: /home/agustine/Downloads/IA_PREDICTIVA/data/03_primary/dataset_modelo_sin_nan_clima.xlsx
